<a href="https://colab.research.google.com/github/KarthikeyanReddy11/23CSBTB29/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

class MDP:
    def __init__(self, states, actions, transition_prob, rewards, gamma=0.9):
        self.states = states
        self.actions = actions
        self.P = transition_prob   # dict: P[s][a] -> list of (prob, next_state)
        self.R = rewards           # dict: R[s][a][s'] -> reward
        self.gamma = gamma

    def value_iteration(self, theta=1e-6):
        V = np.zeros(len(self.states))
        while True:
            delta = 0
            for s in self.states:
                v = V[s]
                Q = []
                for a in self.actions:
                    q = sum([p * (self.R[s][a][s_next] + self.gamma * V[s_next])
                             for (p, s_next) in self.P[s][a]])
                    Q.append(q)
                V[s] = max(Q)
                delta = max(delta, abs(v - V[s]))
            if delta < theta:
                break
        # extract policy
        policy = np.zeros(len(self.states), dtype=int)
        for s in self.states:
            Q = []
            for a in self.actions:
                q = sum([p * (self.R[s][a][s_next] + self.gamma * V[s_next])
                         for (p, s_next) in self.P[s][a]])
                Q.append(q)
            policy[s] = np.argmax(Q)
        return V, policy

    def policy_iteration(self):
        policy = np.random.choice(self.actions, size=len(self.states))
        V = np.zeros(len(self.states))

        while True:
            # Policy evaluation
            while True:
                delta = 0
                for s in self.states:
                    v = V[s]
                    a = policy[s]
                    V[s] = sum([p * (self.R[s][a][s_next] + self.gamma * V[s_next])
                                for (p, s_next) in self.P[s][a]])
                    delta = max(delta, abs(v - V[s]))
                if delta < 1e-6:
                    break

            # Policy improvement
            policy_stable = True
            for s in self.states:
                old_action = policy[s]
                Q = []
                for a in self.actions:
                    q = sum([p * (self.R[s][a][s_next] + self.gamma * V[s_next])
                             for (p, s_next) in self.P[s][a]])
                    Q.append(q)
                policy[s] = np.argmax(Q)
                if old_action != policy[s]:
                    policy_stable = False
            if policy_stable:
                break

        return V, policy


# Example: Simple 3-state MDP
states = [0, 1, 2]
actions = [0, 1]  # 0 = left, 1 = right

# Transition probabilities
P = {
    0: {0: [(1.0, 0)], 1: [(1.0, 1)]},
    1: {0: [(1.0, 0)], 1: [(1.0, 2)]},
    2: {0: [(1.0, 1)], 1: [(1.0, 2)]},
}

# Rewards
R = {
    0: {0: {0: 0}, 1: {1: 0}},
    1: {0: {0: 0}, 1: {2: 1}},
    2: {0: {1: 0}, 1: {2: 0}},
}

mdp = MDP(states, actions, P, R, gamma=0.9)

V_vi, policy_vi = mdp.value_iteration()
V_pi, policy_pi = mdp.policy_iteration()

print("Value Iteration: Values =", V_vi, "Policy =", policy_vi)
print("Policy Iteration: Values =", V_pi, "Policy =", policy_pi)


Value Iteration: Values = [4.73683861 5.26315475 4.73683927] Policy = [1 1 0]
Policy Iteration: Values = [4.73683861 5.26315475 4.73683927] Policy = [1 1 0]
